In [1]:
!pip install pyspark

In [2]:
from pyspark.sql.functions import col,count,sum

In [3]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, ntile

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

output_dir = 'unzipped_data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

zip_file_path = '/content/drive/MyDrive/dataset/archive.zip'
!unzip -q '{zip_file_path}' -d '{output_dir}'

replace unzipped_data/faker_employee.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [6]:
print(f"Files unzipped to: {output_dir}")
print(os.listdir(output_dir))

Files unzipped to: unzipped_data
['faker_employee.csv']


In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark practice").getOrCreate()

In [8]:
file_path = os.path.join(output_dir, 'faker_employee.csv')
df = spark.read.csv(file_path, header=True, inferSchema=True)

df.printSchema()
df.show()

root
 |-- Employee ID: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Joining Date: date (nullable = true)
 |-- Email ID: string (nullable = true)
 |-- Address: string (nullable = true)

+------------------+----------+-----------+--------------------+------+------------+--------------------+--------------------+
|       Employee ID|First Name|  Last Name|          Department|Salary|Joining Date|            Email ID|             Address|
+------------------+----------+-----------+--------------------+------+------------+--------------------+--------------------+
|                 1|  Nicholas|     Martin|               Sales| 50819|  2022-11-16|dannymays@example...|14632 Ashley Traf...|
|  Lake Katrinaside| MH 96450"|       NULL|                NULL|  NULL|        NULL|                NULL|                NULL|
|                 2|       Joy|

In [9]:
df.printSchema()

root
 |-- Employee ID: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Joining Date: date (nullable = true)
 |-- Email ID: string (nullable = true)
 |-- Address: string (nullable = true)



In [10]:
df.describe()

DataFrame[summary: string, Employee ID: string, First Name: string, Last Name: string, Department: string, Salary: string, Email ID: string, Address: string]

In [11]:
from pyspark.sql.functions import col
df = df.withColumn('Salary', col('Salary').cast('float'))

In [12]:
df.describe()

DataFrame[summary: string, Employee ID: string, First Name: string, Last Name: string, Department: string, Salary: string, Email ID: string, Address: string]

In [13]:
df.printSchema()

root
 |-- Employee ID: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: float (nullable = true)
 |-- Joining Date: date (nullable = true)
 |-- Email ID: string (nullable = true)
 |-- Address: string (nullable = true)



In [14]:
df.select('Salary').describe().show()

+-------+-----------------+
|summary|           Salary|
+-------+-----------------+
|  count|          1000000|
|   mean|     64995.810636|
| stddev|8661.449200686111|
|    min|          50000.0|
|    max|          80000.0|
+-------+-----------------+



In [15]:
df.filter(col('Salary') >= 80000).count()

41

In [16]:
df = df.withColumn(('Salary'),col('Salary').cast('float'))
df.printSchema()

root
 |-- Employee ID: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: float (nullable = true)
 |-- Joining Date: date (nullable = true)
 |-- Email ID: string (nullable = true)
 |-- Address: string (nullable = true)



In [17]:
from pyspark.sql.functions import stddev, col
df.select(stddev(col('Salary'))).show()

+-----------------+
|   stddev(Salary)|
+-----------------+
|8661.449200686111|
+-----------------+



In [18]:
df.show()

+------------------+----------+-----------+--------------------+-------+------------+--------------------+--------------------+
|       Employee ID|First Name|  Last Name|          Department| Salary|Joining Date|            Email ID|             Address|
+------------------+----------+-----------+--------------------+-------+------------+--------------------+--------------------+
|                 1|  Nicholas|     Martin|               Sales|50819.0|  2022-11-16|dannymays@example...|14632 Ashley Traf...|
|  Lake Katrinaside| MH 96450"|       NULL|                NULL|   NULL|        NULL|                NULL|                NULL|
|                 2|       Joy|     Miller|               Legal|58024.0|  2023-02-03|heather59@example...|00670 Oliver Harbors|
|Port Andrewchester| TN 14223"|       NULL|                NULL|   NULL|        NULL|                NULL|                NULL|
|                 3|    Thomas|       Roth|                 R&D|54572.0|  2021-08-18|garyhogan@example..

In [19]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn(
    "Email ID",
    regexp_replace(col("Email ID"), "@", "")
)

df.show()

+------------------+----------+-----------+--------------------+-------+------------+--------------------+--------------------+
|       Employee ID|First Name|  Last Name|          Department| Salary|Joining Date|            Email ID|             Address|
+------------------+----------+-----------+--------------------+-------+------------+--------------------+--------------------+
|                 1|  Nicholas|     Martin|               Sales|50819.0|  2022-11-16|dannymaysexample.com|14632 Ashley Traf...|
|  Lake Katrinaside| MH 96450"|       NULL|                NULL|   NULL|        NULL|                NULL|                NULL|
|                 2|       Joy|     Miller|               Legal|58024.0|  2023-02-03|heather59example.net|00670 Oliver Harbors|
|Port Andrewchester| TN 14223"|       NULL|                NULL|   NULL|        NULL|                NULL|                NULL|
|                 3|    Thomas|       Roth|                 R&D|54572.0|  2021-08-18|garyhoganexample.co

In [22]:
df.select('Email ID').show(1)

+--------------------+
|            Email ID|
+--------------------+
|dannymaysexample.com|
+--------------------+
only showing top 1 row


In [25]:
df = df.withColumn('Email ID' ,
regexp_replace(col('Email ID') , 'example' , '@example') )

In [28]:
df.select('Email ID').show(3)

+--------------------+
|            Email ID|
+--------------------+
|dannymays@example...|
|                NULL|
|heather59@example...|
+--------------------+
only showing top 3 rows


In [29]:
df.printSchema()

root
 |-- Employee ID: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: float (nullable = true)
 |-- Joining Date: date (nullable = true)
 |-- Email ID: string (nullable = true)
 |-- Address: string (nullable = true)



In [34]:
from pyspark.sql.functions import col, count, min, max
df.select(count(col("Joining Date")).alias("count"),
          min(col("Joining Date")).alias("min_date"),
          max(col("Joining Date")).alias("max_date")).show()

+-------+----------+----------+
|  count|  min_date|  max_date|
+-------+----------+----------+
|1000000|2020-11-25|2023-11-25|
+-------+----------+----------+



In [37]:

df.select(max(col("Joining Date"))).show()

+-----------------+
|max(Joining Date)|
+-----------------+
|       2023-11-25|
+-----------------+



In [38]:
df.select(max(col('Salary'))).show()

+-----------+
|max(Salary)|
+-----------+
|    80000.0|
+-----------+



In [41]:
from pyspark.sql.functions import col, max, dayofmonth, dayofweek

df.select(dayofweek(max(col('Joining Date'))).alias('max_day')).show()

+-------+
|max_day|
+-------+
|      7|
+-------+



In [44]:
from pyspark.sql.functions import col, datediff , min , max , months_between,lit
df.select(datediff(max(col('Joining Date')), min(col('Joining Date'))).alias('diff')).show()


+----+
|diff|
+----+
|1095|
+----+



In [46]:
df.select(months_between(lit("2025-09-24"), col("Joining Date"))).show()

+----------------------------------------------+
|months_between(2025-09-24, Joining Date, true)|
+----------------------------------------------+
|                                   34.25806452|
|                                          NULL|
|                                   31.67741935|
|                                          NULL|
|                                   49.19354839|
|                                          NULL|
|                                   35.67741935|
|                                          NULL|
|                                   37.25806452|
|                                          NULL|
|                                   48.58064516|
|                                          NULL|
|                                   33.61290323|
|                                          NULL|
|                                   42.87096774|
|                                          NULL|
|                                   55.41935484|
|                   

In [47]:
df.printSchema()

root
 |-- Employee ID: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: float (nullable = true)
 |-- Joining Date: date (nullable = true)
 |-- Email ID: string (nullable = true)
 |-- Address: string (nullable = true)



In [59]:
from pyspark.sql.functions import array
df.select(array(col('First Name'), col('Last Name')).alias('Full_Name')).show(truncate=False)
full_name_df = df.select(
    array(col("First Name"), col("Last Name")).alias("Full Name")
)

full_name_df.show(truncate=False)

+------------------------+
|Full_Name               |
+------------------------+
|[Nicholas, Martin]      |
|[ MH 96450", NULL]      |
|[Joy, Miller]           |
|[ TN 14223", NULL]      |
|[Thomas, Roth]          |
|[ AR 97982", NULL]      |
|[Brittney, Morgan]      |
|[ MA 16312", NULL]      |
|[Katherine, Bell]       |
|[ CT 32200", NULL]      |
|[Angela, Mullins]       |
|[ WY 61052", NULL]      |
|[Stephanie, Christensen]|
|[ VI 54818", NULL]      |
|[Jesse, Garrett]        |
|[ IA 97716", NULL]      |
|[Juan, Wade]            |
|[NULL, NULL]            |
|[Danielle, Gonzalez]    |
|[ RI 09697", NULL]      |
+------------------------+
only showing top 20 rows
+------------------------+
|Full Name               |
+------------------------+
|[Nicholas, Martin]      |
|[ MH 96450", NULL]      |
|[Joy, Miller]           |
|[ TN 14223", NULL]      |
|[Thomas, Roth]          |
|[ AR 97982", NULL]      |
|[Brittney, Morgan]      |
|[ MA 16312", NULL]      |
|[Katherine, Bell]       |
|[ 

In [56]:
type(a)

NoneType